In [1]:
import time
import numpy as np

from optimizers.tilde_pid import exact_tilde_union_info_minimizer
from optimizers.thin_pid import exact_thin_pid_minimizer
from utils import generate_randomBC

In [2]:
step=100
start=100
max_dX=1000
num_runs=1  # number of runs for each configuration
ratio=[0.6, 0.1]

In [3]:
# Generate dx, dy, dT sequence
R = np.array([1, ratio[0]])
sequence = np.arange(start, max_dX + 1, step)
XY_dim = np.array([seq * R for seq in sequence]).astype(int)  # TXY_dim = sequence * R = [dT, dx, dy]
print(XY_dim)

[[ 100   60]
 [ 200  120]
 [ 300  180]
 [ 400  240]
 [ 500  300]
 [ 600  360]
 [ 700  420]
 [ 800  480]
 [ 900  540]
 [1000  600]]


In [4]:
## Case 2: fix dY = 100
X_dim = np.arange(start, max_dX + 1, step)
Y_dim = np.ones(len(sequence), dtype=int) * 100
print(X_dim, Y_dim)

[ 100  200  300  400  500  600  700  800  900 1000] [100 100 100 100 100 100 100 100 100 100]


In [5]:
dtype = [('dT', int), ('dx', int), ('dy', int), ('obj_tilde', float), ('obj_thin', float),
             ('execution_time_tilde', float), ('execution_time_thin', float), ('itr_tilde', int),
             ('itr_thin', int)]
result_comparison = np.zeros((len(sequence), num_runs), dtype=dtype)  # compare tilde and thin pid

In [6]:
for i, seq in enumerate(sequence):
    dx, dy = XY_dim[i]
    dT = 1
    print(f"\nRunning config [dx, dy, dT]: {dx:3d}-{dy:3d}-{dT:3d}")

    for j in range(num_runs):

        hx, hy = generate_randomBC(dT, dx, dy)
        if dx + dy < 2000:  # only run the original algorithm when the size is small, otherwise it takes too long
            start_time1 = time.time()
            sig_tilde, obj_tilde, itr_tilde, _ = exact_tilde_union_info_minimizer(hx, hy, ret_obj=True, reg=1e-8)
            end_time1 = time.time()
        else:
            sig_tilde, obj_tilde, itr_tilde = 0, 0, 0
            end_time1, start_time1 = 0.0, 0.0
            print(f"Skipping tilde pid for {dx:3d}-{dy:3d}-{dT:3d}")

        start_time2 = time.time()
        sig_thin, obj_thin, itr_thin, _ = exact_thin_pid_minimizer(hx, hy, ret_obj=True, reg=1e-8)
        end_time2 = time.time()

        execution_time_tilde = end_time1 - start_time1
        execution_time_thin = end_time2 - start_time2
        result_comparison[i, j] = (
            dT, dx, dy, obj_tilde, obj_thin, execution_time_tilde, execution_time_thin, itr_tilde, itr_thin
        )

    # print average results
    print(f"Average obj_tilde: {np.mean(result_comparison[i]['obj_tilde']):4.9f}, obj_thin: {np.mean(result_comparison[i]['obj_thin']):4.9f}, time_tilde: {np.mean(result_comparison[i]['execution_time_tilde']):3.4f}, time_thin: {np.mean(result_comparison[i]['execution_time_thin']):3.4f}")


Running config [dx, dy, dT]: 100- 60-  1
Average obj_tilde: 3.340458047, obj_thin: 3.340458055, time_tilde: 0.7767, time_thin: 0.1129

Running config [dx, dy, dT]: 200-120-  1
Average obj_tilde: 3.834549090, obj_thin: 3.834549097, time_tilde: 3.1331, time_thin: 0.3053

Running config [dx, dy, dT]: 300-180-  1
Average obj_tilde: 4.183628411, obj_thin: 4.183628418, time_tilde: 9.0116, time_thin: 1.0353

Running config [dx, dy, dT]: 400-240-  1
Average obj_tilde: 4.347697398, obj_thin: 4.347697405, time_tilde: 18.0147, time_thin: 2.2073

Running config [dx, dy, dT]: 500-300-  1
Average obj_tilde: 4.496460701, obj_thin: 4.496460708, time_tilde: 34.7842, time_thin: 3.5542

Running config [dx, dy, dT]: 600-360-  1
Average obj_tilde: 4.587129534, obj_thin: 4.587129541, time_tilde: 54.0085, time_thin: 5.7485

Running config [dx, dy, dT]: 700-420-  1
Average obj_tilde: 4.681350356, obj_thin: 4.681350363, time_tilde: 82.7852, time_thin: 8.2091

Running config [dx, dy, dT]: 800-480-  1
Average o